# Average Degree of Nearest Neighbors
## For undirected networks

### Imports

In [1]:
from __future__ import annotations
import networkx as nx
import numpy as np
import pytest
from abc import ABC, abstractmethod

### Error Classes

In [2]:
class NormalizationError(Exception):
    """Exception raised for errors in the normalization.

    Attributes:
        message -- explanation of the error
    """

    def __init__(self, message="An error occurred during normalization."):
        self.message = message
        super().__init__(self.message)

class NullGraphError(Exception):
    """Exception raised for null graph."""
    pass

class EmptyGraphError(Exception):
    """Exception raised for empty graph. Nodes with no edges."""
    pass

def check_raw_value(func):
    """Decorator to check if raw value is None. If it is, raise an error."""
    def wrapper(self, *args):
        if self._raw_value is not None:
            return func(self, *args)
        else:
            raise ValueError("Raw value is None. Call compute() method first.")
    return wrapper

### Abstract Class

In [3]:
class _Property(ABC):
    """Abstract base class for all properties."""
    _return_type = None
    _use_paths = False
    _use_direction = False
    _use_selfloops = False
    _use_giant_component = False

    def __init__(self, G: nx.DiGraph):
        self.G = G
        self._raw_value = None
        self._n_nodes = self.G.number_of_nodes()
        if self._n_nodes == 0:
            raise NullGraphError("A null graph has no Average Out Degree.")

    @abstractmethod
    def compute(self):
        return self._raw_value

    @abstractmethod
    def norm_biol(self, *args):
        pass

    @abstractmethod
    def norm_network(self, *args):
        pass

### Decorators

In [4]:
def use_direction(cls):
    cls._use_direction = True
    return cls

def use_selfloops(cls):
    cls._use_selfloops = True
    return cls

def use_giant_component(cls):
    cls._use_giant_component = True
    return cls

def return_scalar(cls):
    cls._return_type = "scalar"
    return cls

def return_distribution(cls):
    cls._return_type = "distribution"
    return cls

def use_paths(cls):
    cls._use_paths = True
    return cls

### Auxiliar Fxns

In [26]:
def remove_self_loops(G: nx.DiGraph, n_nodes):
    G.remove_edges_from([(i,i) for i in range(n_nodes)])
    return G

### Average Degree of Nearest Neighbors Class

In [38]:
@return_distribution 
class AverageDegreeNearestNeighbors(_Property): # Hereda de la clase _Property
    """Average Degree of Nearest Neighbors.

    The Average Degree of Nearest Neighbors here is considered for an undirected network, meaning the neighborhood for each node
    is all nodes it has a connection with, regardless of direction.

    Methods:
        compute: Compute the average degree of nearest neighbors.
        norm_biol: NO IMPLEMENTATION.
        norm_network: Normalize the average degree of nearest neighbors to all nodes in the graph.
    """
    __name__ = 'Average Degree for Nearest Neighbors (Undirected)'

    def __init__(self, G: nx.DiGraph):
        """
        Args:
            G (nx.DiGraph): Graph.
        """
        super().__init__(G)

    def compute(self) -> np.array:
        """Compute the density of the graph.

        Returns:
            float: Density of the graph.    
        """
        no_self_loops = nx.DiGraph
        no_self_loops = remove_self_loops(self.G, self._n_nodes)
        dict_av_degree = nx.average_neighbor_degree(no_self_loops.to_undirected())
        self._raw_value = np.fromiter(dict_av_degree.values(), dtype=float)

        return self._raw_value

    @check_raw_value    # Decorator to check if raw value is None. If it is, raise an error.
    def norm_biol(self):
        raise NormalizationError("No biological normalization implemented.")

    @check_raw_value
    def norm_network(self) -> np.array:
        """Normalize the average degree for nearest neighbors to the number of nodes (-1 because you can not be your own neighbor)"""
        return self._raw_value * (1 / (self._n_nodes - 1))

### Testing

In [39]:
# Null graph
G = nx.DiGraph()

with pytest.raises(NullGraphError) as e_info:
    property = AverageDegreeNearestNeighbors(G)

# Empty graph, allows instance from an empty graph, but does not compute
n_nodes= 5
G.add_nodes_from(range(n_nodes))
property = AverageDegreeNearestNeighbors(G)

assert not np.any(property.compute()) #All 0s

# add edges
# complete directed graph with self loops
G.add_edges_from([(i, j) for i in range(n_nodes) for j in range(n_nodes)])
property = AverageDegreeNearestNeighbors(G)
expected = np.array([4., 4., 4., 4., 4.])
expected_1 = np.array([1., 1., 1., 1., 1.])

assert np.allclose(property.compute(), expected)
assert np.allclose(property.norm_network(), expected_1)
with pytest.raises(NormalizationError) as e_info:
    property.norm_biol()

# add edges
# only half of the nodes are parents and regulate every node in the graph
G = nx.DiGraph()
n_nodes= 5
G.add_nodes_from(range(n_nodes))
G.add_edges_from([(i, j) for i in range(n_nodes//2) for j in range(n_nodes)])
property = AverageDegreeNearestNeighbors(G)
expected = np.array([2.5, 2.5, 4. , 4. , 4. ])
expected_1 = np.array([0.625, 0.625, 1.   , 1.   , 1.   ])

assert np.allclose(property.compute(), expected)
assert np.allclose(property.norm_network(), expected_1)
with pytest.raises(NormalizationError) as e_info:
    property.norm_biol()